# Demand & Available Cabs — Model comparison notebook

This notebook shows how to train **LSTM, RandomForest, CatBoost, LightGBM, XGBoost, Prophet** (where available),
compare their evaluation metrics, pick the best model by **accuracy** (defined below), then use the chosen model
to forecast future **demand** and **available_cabs**, and compute **shortage = demand - available_cabs**.

Place your CSV at: `/mnt/data/Final_Processed.csv` (your path: `C:\Users\adity\Downloads\MMDS\Project\Final_Processed.csv`).

Input dates must be in `dd mm yyyy` format when asked for a forecast horizon.

Notes:
- This notebook uses safe checks for optional libraries (catboost, xgboost, lightgbm, prophet). If a library is missing,
  its model step will be skipped and you will be prompted to install it in your environment.
- For regression 'accuracy' we compute the fraction of predictions within ±10% of true value (configurable).
- LSTM model uses Keras (TensorFlow). If running on CPU, training could take time; reduce epochs for quick tests.


In [5]:
# CELL: Imports and utility functions
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def accuracy_within_tol(y_true, y_pred, tol=0.10):
    # fraction of samples where |pred - true| <= tol * true
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    # avoid division by zero: where y_true==0 treat as absolute tolerance of tol
    denom = np.where(np.abs(y_true) < 1e-9, 1.0, np.abs(y_true))
    within = np.abs(y_pred - y_true) <= tol * denom
    return np.mean(within)

def evaluate_regression(y_true, y_pred, tol=0.10):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': rmse(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
        'Accuracy(±{}%)'.format(int(tol*100)): accuracy_within_tol(y_true, y_pred, tol)
    }

print('imports ready')


imports ready


In [6]:
# CELL: Load dataset
csv_path = r"C:\Users\adity\Downloads\MMDS\Project\Final_Processed.csv"

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        "CSV not found at C:\\Users\\adity\\Downloads\\MMDS\\Project\\Final_Processed.csv. "
        "Please place Final_Processed.csv there."
    )

df = pd.read_csv(csv_path)
print("rows, cols =", df.shape)
df.head()


rows, cols = (347988, 14)


,ride_id,datetime,ride_status,pickup_area,drop_area,year,month,day,hour,date,time_zone,demand,available_cabs,shortage
0,BKG002834,2024-01-01T14:59:00.000Z,Success,Anekal,Bellandur,2024,1,1,14,2024-01-01,midday,16,24,-8
1,BKG004040,2024-01-01T12:47:00.000Z,Cancelled by Driver,Anekal,Yeshwanthpur,2024,1,1,12,2024-01-01,midday,16,24,-8
2,BKG034183,2024-01-01T14:38:00.000Z,Success,Anekal,RT Nagar,2024,1,1,14,2024-01-01,midday,16,24,-8
3,BKG042360,2024-01-01T13:34:00.000Z,Success,Anekal,Kengeri,2024,1,1,13,2024-01-01,midday,16,24,-8
4,BKG059450,2024-01-01T14:30:00.000Z,Cancelled by Driver,Anekal,Magadi Road,2024,1,1,14,2024-01-01,midday,16,24,-8


In [7]:
# CELL: Basic preprocessing & feature engineering
df['datetime'] = pd.to_datetime(df['datetime'])
# ensure date column exists
if 'date' not in df.columns:
    df['date'] = df['datetime'].dt.date

# we'll create features for the supervised models (non-sequence models)
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['day'] = df['datetime'].dt.day
df['hour'] = df['datetime'].dt.hour
df['weekday'] = df['datetime'].dt.weekday

# Drop rows with NA in targets
df = df.dropna(subset=['demand', 'available_cabs'])
df = df.reset_index(drop=True)
df.shape


(347988, 15)

## Modeling approach

We'll train separate models for the two targets: `demand` and `available_cabs`.

For tree-based / ML models we'll use the following features:
- numeric calendar features: year, month, day, hour, weekday
- pickup_area (one-hot or label encoded)
- time_zone (label encoded) — if present

For LSTM we will train a simple sequence model using a sliding window of past demand (and optionally exogenous features). For simplicity this notebook trains an LSTM on the univariate series (past demand -> next demand) and similarly for available_cabs.


In [8]:
# CELL: Prepare features for classic ML models
feature_cols = ['year','month','day','hour','weekday']
if 'pickup_area' in df.columns:
    # simple label encoding
    df['pickup_area_le'] = df['pickup_area'].astype('category').cat.codes
    feature_cols.append('pickup_area_le')
if 'time_zone' in df.columns:
    df['time_zone_le'] = df['time_zone'].astype('category').cat.codes
    feature_cols.append('time_zone_le')

feature_cols


['year', 'month', 'day', 'hour', 'weekday', 'pickup_area_le', 'time_zone_le']

In [9]:
# CELL: Train-test split (time-aware) — we'll hold out the last 20% of time for test
df_sorted = df.sort_values('datetime').reset_index(drop=True)
split_idx = int(len(df_sorted) * 0.8)
train_df = df_sorted.iloc[:split_idx].copy()
test_df = df_sorted.iloc[split_idx:].copy()

print('train:', train_df.shape, 'test:', test_df.shape)


train: (278390, 17) test: (69598, 17)


In [10]:
# CELL: Function to train a set of models for one target
def train_and_evaluate(target, feature_cols, train_df, test_df, models_to_run=None, tol=0.10):
    """
    models_to_run: list of strings from ['random_forest','xgboost','lightgbm','catboost','lstm','prophet']
    returns: dict of results and trained models
    """
    results = {}
    trained_models = {}
    X_train = train_df[feature_cols].values
    X_test = test_df[feature_cols].values
    y_train = train_df[target].values
    y_test = test_df[target].values

    if models_to_run is None:
        models_to_run = ['random_forest','xgboost','lightgbm','catboost','lstm','prophet']

    # Random Forest (sklearn)
    if 'random_forest' in models_to_run:
        rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)
        results['random_forest'] = evaluate_regression(y_test, y_pred, tol=tol)
        trained_models['random_forest'] = rf
        print('RF done')

    # XGBoost
    if 'xgboost' in models_to_run:
        try:
            import xgboost as xgb
            xg = xgb.XGBRegressor(tree_method='hist', n_estimators=200, random_state=42, n_jobs=-1)
            xg.fit(X_train, y_train)
            y_pred = xg.predict(X_test)
            results['xgboost'] = evaluate_regression(y_test, y_pred, tol=tol)
            trained_models['xgboost'] = xg
            print('XGB done')
        except Exception as e:
            print('XGBoost not available or failed:', e)

    # LightGBM
    if 'lightgbm' in models_to_run:
        try:
            import lightgbm as lgb
            lg = lgb.LGBMRegressor(n_estimators=200, random_state=42, n_jobs=-1)
            lg.fit(X_train, y_train)
            y_pred = lg.predict(X_test)
            results['lightgbm'] = evaluate_regression(y_test, y_pred, tol=tol)
            trained_models['lightgbm'] = lg
            print('LightGBM done')
        except Exception as e:
            print('LightGBM not available or failed:', e)

    # CatBoost
    if 'catboost' in models_to_run:
        try:
            from catboost import CatBoostRegressor
            cb = CatBoostRegressor(verbose=0, random_state=42, iterations=500)
            cb.fit(X_train, y_train)
            y_pred = cb.predict(X_test)
            results['catboost'] = evaluate_regression(y_test, y_pred, tol=tol)
            trained_models['catboost'] = cb
            print('CatBoost done')
        except Exception as e:
            print('CatBoost not available or failed:', e)

    # LSTM (univariate sequence) - we'll use Keras
    if 'lstm' in models_to_run:
        try:
            from tensorflow.keras.models import Sequential
            from tensorflow.keras.layers import LSTM, Dense
            from tensorflow.keras.callbacks import EarlyStopping
            # we'll train on the univariate series using past N lags
            def make_supervised_series(series, n_lags=24):
                X, y = [], []
                for i in range(n_lags, len(series)):
                    X.append(series[i-n_lags:i])
                    y.append(series[i])
                return np.array(X), np.array(y)

            series_train = y_train
            series_test = np.concatenate([y_train[-24:], y_test])  # warm-start with last window
            n_lags = 24
            Xs_train, ys_train = make_supervised_series(series_train, n_lags=n_lags)
            Xs_test, ys_test = make_supervised_series(series_test, n_lags=n_lags)
            # reshape for LSTM: (samples, timesteps, features)
            Xs_train = Xs_train.reshape((Xs_train.shape[0], Xs_train.shape[1], 1))
            Xs_test = Xs_test.reshape((Xs_test.shape[0], Xs_test.shape[1], 1))

            model = Sequential([
                LSTM(64, input_shape=(n_lags,1)),
                Dense(32, activation='relu'),
                Dense(1)
            ])
            model.compile(optimizer='adam', loss='mse')
            es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
            model.fit(Xs_train, ys_train, validation_split=0.1, epochs=20, batch_size=32, callbacks=[es], verbose=0)
            y_pred = model.predict(Xs_test).flatten()
            results['lstm'] = evaluate_regression(ys_test, y_pred, tol=tol)
            trained_models['lstm'] = model
            print('LSTM done')
        except Exception as e:
            print('LSTM training failed or tensorflow not available:', e)

    # Prophet (univariate TS forecasting)
    if 'prophet' in models_to_run:
        try:
            from prophet import Prophet
            # Train Prophet on train_df aggregated by datetime (ds, y)
            df_prop = train_df[['datetime', target]].rename(columns={'datetime':'ds', target:'y'})
            m = Prophet()
            m.fit(df_prop)
            # make predictions for test_df datetimes
            future = test_df[['datetime']].rename(columns={'datetime':'ds'})
            forecast = m.predict(future)
            y_pred = forecast['yhat'].values
            results['prophet'] = evaluate_regression(y_test, y_pred, tol=tol)
            trained_models['prophet'] = m
            print('Prophet done')
        except Exception as e:
            print('Prophet not available or failed:', e)

    return results, trained_models

print('training helper ready')


training helper ready


In [11]:
# CELL: Run training for both targets (this may take time). You can control which models to run.
models_to_run = ['random_forest','xgboost','lightgbm','catboost','lstm','prophet']
results_all = {}
trained_all = {}
for target in ['demand','available_cabs']:
    print('\n---- Training for target:', target, '----')
    res, tmodels = train_and_evaluate(target, feature_cols, train_df, test_df, models_to_run=models_to_run, tol=0.10)
    results_all[target] = res
    trained_all[target] = tmodels
    print('Results for', target, ':', res)

import pandas as pd
summary_rows = []
for target, resdict in results_all.items():
    for mname, metrics in resdict.items():
        row = {'target':target, 'model':mname}
        row.update(metrics)
        summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

# Save trained models that exist (skipping LSTM and Prophet model objects if heavy)
os.makedirs('/mnt/data/trained_models', exist_ok=True)
for target, tdict in trained_all.items():
    for name, model in tdict.items():
        try:
            joblib.dump(model, f'/mnt/data/trained_models/{target}_{name}.pkl')
        except Exception:
            # some objects (like TensorFlow models or Prophet) may not be joblib-dumpable
            try:
                model.save(f'/mnt/data/trained_models/{target}_{name}_tf')
            except Exception:
                pass
print('Done — models saved where possible')



---- Training for target: demand ----
RF done
XGB done
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002142 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 329
[LightGBM] [Info] Number of data points in the train set: 278390, number of used features: 6
[LightGBM] [Info] Start training from score 16.631377
LightGBM done
CatBoost done
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 28s 13ms/step
LSTM done
Prophet not available or failed: Column ds has timezone specified, which is not supported. Remove timezone.
Results for demand : {'random_forest': {'MAE': 2.665515819420098, 'RMSE': np.float64(3.807543583716492), 'R2': 0.5422002198061548, 'Accuracy(±10%)': np.float64(0.2708985890399149)}, 'xgboost': {'MAE': 2.745870590209961, 'RMSE': np.float64(3.79562822130413), 'R2': 0.5450609922409058, 'Accuracy(±10%)': np.float64(0.18372654386620305)}, 'lightgbm':

,target,model,MAE,RMSE,R2,Accuracy(±10%)
0,demand,random_forest,2.665516,3.807544,0.542200,0.270899
1,demand,xgboost,2.745871,3.795628,0.545061,0.183727
2,demand,lightgbm,2.769142,3.623680,0.585346,0.183482
3,demand,catboost,2.713686,3.617706,0.586712,0.157490
4,demand,lstm,3.202015,4.375675,0.395389,0.139688
5,available_cabs,random_forest,2.791152,3.999685,0.488436,0.241142
6,available_cabs,xgboost,2.787068,3.810297,0.535735,0.161959
7,available_cabs,lightgbm,2.739711,3.567491,0.593019,0.165637
8,available_cabs,catboost,2.584100,3.422650,0.625395,0.165249
9,available_cabs,lstm,3.163445,4.333936,0.399361,0.134199


Done — models saved where possible


## Selecting best model and forecasting future

We'll pick the best model per target using the chosen accuracy metric (Accuracy ±10%). After selecting, we'll ask for a forecast end date (dd mm yyyy) and produce predictions from the last known datetime up to that date (hourly frequency). Finally we compute shortage = demand - available_cabs.


In [15]:
# CELL: Choose best model per target and forecast
def choose_best_model_name(results_for_target, metric_name='Accuracy(±10%)'):
    # pick model with highest metric_name
    best_name = None
    best_val = -np.inf
    for m, metrics in results_for_target.items():
        if metric_name in metrics and metrics[metric_name] > best_val:
            best_val = metrics[metric_name]
            best_name = m
    return best_name, best_val

best_models = {}
for target, resdict in results_all.items():
    name, val = choose_best_model_name(resdict, metric_name='Accuracy(±10%)')
    best_models[target] = name
    print('Best for', target, '->', name, 'with accuracy', val)

# Forecasting function (supports rf/xgb/lgb/catboost using features, lstm and prophet if available)
def forecast_future(target, best_model_name, trained_models, last_datetime, end_date_str, freq='H'):
    """
    end_date_str: in 'dd mm yyyy'
    """
    # parse end date
    end_dt = pd.to_datetime(end_date_str, format='%d %m %Y')
    # build index from last_datetime + 1 hour to end_dt inclusive
    start = pd.to_datetime(last_datetime) + pd.Timedelta(hours=1)
    if end_dt < start:
        raise ValueError('end date must be after last known datetime')
    future_index = pd.date_range(start=start, end=end_dt, freq=freq)
    print('Forecasting', len(future_index), 'steps from', start, 'to', end_dt)
    
    # If model is tree-based and uses feature_cols, we need to construct feature rows for each future timestamp
    if best_model_name in ['random_forest','xgboost','lightgbm','catboost']:
        rows = []
        for dt in future_index:
            row = {
                'year': dt.year, 'month': dt.month, 'day': dt.day, 'hour': dt.hour, 'weekday': dt.weekday()
            }
            # for categorical features, use mode from training set
            if 'pickup_area_le' in feature_cols:
                row['pickup_area_le'] = train_df['pickup_area_le'].mode().iloc[0]
            if 'time_zone_le' in feature_cols:
                row['time_zone_le'] = train_df['time_zone_le'].mode().iloc[0]
            rows.append(row)
        X_future = pd.DataFrame(rows)[feature_cols].values
        model = trained_models.get(best_model_name)
        y_future = model.predict(X_future)
        return pd.DataFrame({'datetime':future_index, target:y_future})

    if best_model_name == 'lstm':
        # simple recursive forecasting using the last n_lags from full series
        try:
            model = trained_models['lstm']
            n_lags = 24
            # get the full series up to last_datetime
            full_series = df_sorted[df_sorted['datetime'] <= last_datetime][target].values
            preds = []
            window = list(full_series[-n_lags:])
            for _ in range(len(future_index)):
                x = np.array(window[-n_lags:]).reshape((1,n_lags,1))
                p = model.predict(x).flatten()[0]
                preds.append(p)
                window.append(p)
            return pd.DataFrame({'datetime':future_index, target:preds})
        except Exception as e:
            raise RuntimeError('LSTM forecasting failed: '+str(e))

    if best_model_name == 'prophet':
        try:
            m = trained_models['prophet']
            future_df = pd.DataFrame({'ds': future_index})
            fc = m.predict(future_df)
            return pd.DataFrame({'datetime': future_index, target: fc['yhat'].values})
        except Exception as e:
            raise RuntimeError('Prophet forecasting failed: '+str(e))

    raise NotImplementedError('Model forecasting not implemented for '+str(best_model_name))

print('Forecast helper ready')


Best for demand -> random_forest with accuracy 0.2708985890399149
Best for available_cabs -> random_forest with accuracy 0.24114198683870225
Forecast helper ready


In [16]:
def forecast_future(target, best_model_name, trained_models, last_datetime, end_date_str, freq='H'):
    """
    end_date_str: in 'dd mm yyyy'
    """
    # Convert both to timezone-naive
    last_datetime = pd.to_datetime(last_datetime).tz_localize(None)
    end_dt = pd.to_datetime(end_date_str, format='%d %m %Y').tz_localize(None)

    # Build index from last_datetime + 1 hour to end_dt
    start = last_datetime + pd.Timedelta(hours=1)
    if end_dt < start:
        raise ValueError("end date must be after last known datetime")

    future_index = pd.date_range(start=start, end=end_dt, freq=freq)
    print('Forecasting', len(future_index), 'steps from', start, 'to', end_dt)

    # Tree models (RF/XGB/LightGBM/CatBoost)
    if best_model_name in ['random_forest','xgboost','lightgbm','catboost']:
        rows = []
        for dt in future_index:
            row = {
                'year': dt.year, 'month': dt.month, 'day': dt.day,
                'hour': dt.hour, 'weekday': dt.weekday()
            }
            if 'pickup_area_le' in feature_cols:
                row['pickup_area_le'] = train_df['pickup_area_le'].mode().iloc[0]
            if 'time_zone_le' in feature_cols:
                row['time_zone_le'] = train_df['time_zone_le'].mode().iloc[0]
            rows.append(row)

        X_future = pd.DataFrame(rows)[feature_cols].values
        model = trained_models.get(best_model_name)
        y_future = model.predict(X_future)

        return pd.DataFrame({'datetime': future_index, target: y_future})

    # (LSTM and Prophet handled below)


## Save results and conclusion

The `summary_df` contains evaluation metrics for each model and target. The `fc` dataframe contains the forecasted demand, available_cabs, and computed shortage.

Save `Model_Comparison_and_Forecasting.ipynb` and run in VS Code or JupyterLab. If some models didn't run because packages are missing, install them with pip (in your environment):
- `pip install xgboost lightgbm catboost prophet tensorflow` (or use conda for faster installs)

Tips:
- Reduce LSTM epochs to 5–10 for quick prototyping.
- For production forecasting, consider cross-validation (time-series CV) and hyperparameter tuning.


In [18]:
# =============================
# MODEL COMPARISON TABLE (NO ACCURACY)
# =============================

# Safety check
if "results_all" not in globals():
    raise RuntimeError("❌ ERROR: You must run the model training cell before running this comparison.")

print("🔍 Creating model comparison table...\n")

comparison_rows = []

for target, models_dict in results_all.items():
    for model_name, metrics in models_dict.items():
        row = {
            "Target": target,
            "Model": model_name,
            "MAE": round(metrics["MAE"], 4),
            "RMSE": round(metrics["RMSE"], 4),
            "R2": round(metrics["R2"], 4)
        }
        comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)

# Sort by target, then best RMSE (lower is better)
comparison_df = comparison_df.sort_values(
    by=["Target", "RMSE"], ascending=[True, True]
).reset_index(drop=True)

print("📊 Model Comparison Table:")
display(comparison_df)

# =============================
# BEST MODEL PER TARGET (RMSE-based)
# =============================

best_model_summary = {}

for target in comparison_df["Target"].unique():

    # best = lowest RMSE
    best_row = (
        comparison_df[comparison_df["Target"] == target]
        .sort_values(by="RMSE", ascending=True)
        .iloc[0]
    )

    best_model_summary[target] = {
        "Best Model": best_row["Model"],
        "RMSE": best_row["RMSE"],
        "MAE": best_row["MAE"],
        "R2": best_row["R2"]
    }

print("\nBest Models Selected (Based on Lowest RMSE):")
best_model_summary_df = pd.DataFrame(best_model_summary).T

display(best_model_summary_df)

# Store best models for forecasting
best_models_rmse = {target: info["Best Model"] for target, info in best_model_summary.items()}

print("\n Final chosen models for forecasting:")
print(best_models_rmse)


🔍 Creating model comparison table...

📊 Model Comparison Table:


,Target,Model,MAE,RMSE,R2
0,available_cabs,catboost,2.5841,3.4226,0.6254
1,available_cabs,lightgbm,2.7397,3.5675,0.5930
2,available_cabs,xgboost,2.7871,3.8103,0.5357
3,available_cabs,random_forest,2.7912,3.9997,0.4884
4,available_cabs,lstm,3.1634,4.3339,0.3994
5,demand,catboost,2.7137,3.6177,0.5867
6,demand,lightgbm,2.7691,3.6237,0.5853
7,demand,xgboost,2.7459,3.7956,0.5451
8,demand,random_forest,2.6655,3.8075,0.5422
9,demand,lstm,3.2020,4.3757,0.3954



Best Models Selected (Based on Lowest RMSE):


,Best Model,RMSE,MAE,R2
available_cabs,catboost,3.4226,2.5841,0.6254
demand,catboost,3.6177,2.7137,0.5867



 Final chosen models for forecasting:
{'available_cabs': 'catboost', 'demand': 'catboost'}


In [22]:
# ===============================================
# IMPORTS
# ===============================================
import pandas as pd
import numpy as np
import joblib
import os
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score


# ===============================================
# FIXED train_and_evaluate FUNCTION (CATBOOST ONLY)
# ===============================================
def train_and_evaluate(target, feature_cols, train_df, test_df, models_to_run=['catboost'], tol=0.10):

    # Split features & target
    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()
    y_train = train_df[target]
    y_test = test_df[target]

    # Convert object columns to category (CatBoost requirement)
    cat_cols = X_train.select_dtypes(include=['object']).columns
    for c in cat_cols:
        X_train[c] = X_train[c].astype("category")
        X_test[c] = X_test[c].astype("category")

    results = {}
    trained_models = {}

    # Only CatBoost
    if 'catboost' in models_to_run:
        model = CatBoostRegressor(
            depth=8,
            learning_rate=0.05,
            iterations=800,
            loss_function='RMSE',
            verbose=False
        )

        # Train
        model.fit(X_train, y_train)

        # Predict
        preds = model.predict(X_test)

        # Metrics
        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)
        results['catboost'] = {
            'mae': mae,
            'r2': r2
        }

        trained_models['catboost'] = model

    return results, trained_models



# ===============================================
# MAIN TRAINING LOOP FOR BOTH TARGETS
# ===============================================
models_to_run = ['catboost']
results_all = {}
trained_all = {}

for target in ['demand', 'available_cabs']:
    print(f"\n---- Training for target: {target} ----")
    res, tmodels = train_and_evaluate(
        target,
        feature_cols,
        train_df,
        test_df,
        models_to_run=models_to_run,
        tol=0.10
    )
    results_all[target] = res
    trained_all[target] = tmodels
    print("Results:", res)


# ===============================================
# SUMMARY TABLE
# ===============================================
summary_rows = []
for target, resdict in results_all.items():
    for model_name, metrics in resdict.items():
        row = {'target': target, 'model': model_name}
        row.update(metrics)
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)



# ===============================================
# SAVE TRAINED CATBOOST MODELS
# ===============================================
os.makedirs('/mnt/data/trained_models', exist_ok=True)

for target, models in trained_all.items():
    for name, model in models.items():
        filepath = f'/mnt/data/trained_models/{target}_{name}.cbm'
        model.save_model(filepath)
        print(f"Saved {target} {name} → {filepath}")

print("\n✅ Done — CatBoost models trained and saved successfully.")



---- Training for target: demand ----
Results: {'catboost': {'mae': 2.831506517985305, 'r2': 0.572448264373275}}

---- Training for target: available_cabs ----
Results: {'catboost': {'mae': 2.81570256690344, 'r2': 0.5771851540387164}}


,target,model,mae,r2
0,demand,catboost,2.831507,0.572448
1,available_cabs,catboost,2.815703,0.577185


Saved demand catboost → /mnt/data/trained_models/demand_catboost.cbm
Saved available_cabs catboost → /mnt/data/trained_models/available_cabs_catboost.cbm

✅ Done — CatBoost models trained and saved successfully.


In [23]:
# ========================================================
# IMPORTS
# ========================================================
import pandas as pd
import numpy as np
import os
import joblib
from catboost import CatBoostRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, r2_score



# ========================================================
# FUNCTION: train + hypertune + evaluate
# ========================================================
def train_and_evaluate_catboost(target, feature_cols, train_df, test_df):

    # --------------------------
    # Split features and target
    # --------------------------
    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()
    y_train = train_df[target]
    y_test = test_df[target]

    # --------------------------
    # Convert object -> category
    # --------------------------
    cat_cols = X_train.select_dtypes(include=['object']).columns
    for c in cat_cols:
        X_train[c] = X_train[c].astype("category")
        X_test[c] = X_test[c].astype("category")

    # --------------------------
    # Hyperparameter Search Space
    # --------------------------
    param_dist = {
        "depth": [4, 6, 8, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "iterations": [300, 500, 800, 1200],
        "l2_leaf_reg": [1, 3, 5, 7, 9, 11],
        "bagging_temperature": [0.1, 0.3, 0.5, 1]
    }

    base_model = CatBoostRegressor(
        loss_function='RMSE',
        verbose=False
    )

    # --------------------------
    # RandomizedSearchCV
    # --------------------------
    tuner = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_dist,
        n_iter=20,
        scoring='neg_mean_absolute_error',
        cv=3,
        random_state=42,
        verbose=1
    )

    print(f"🔍 Hyper-tuning CatBoost for {target} ...")
    tuner.fit(X_train, y_train)

    best_model = tuner.best_estimator_
    print("➡ Best Params:", tuner.best_params_)

    # --------------------------
    # Predict
    # --------------------------
    preds = best_model.predict(X_test)

    # --------------------------
    # Metrics
    # --------------------------
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    metrics = {
        "mae": mae,
        "r2": r2,
        "best_params": tuner.best_params_
    }

    return best_model, metrics



# ========================================================
# MAIN LOOP (Demand + Available Cabs)
# ========================================================
results_all = {}
trained_all = {}

for target in ["demand", "available_cabs"]:
    print(f"\n==============================")
    print(f" TRAINING WITH HYPERTUNING → {target}")
    print(f"==============================")

    model, metrics = train_and_evaluate_catboost(
        target,
        feature_cols,
        train_df,
        test_df
    )

    trained_all[target] = model
    results_all[target] = metrics

    print(f"\n📌 Performance for {target}:")
    print("MAE:", metrics['mae'])
    print("R2:", metrics['r2'])



# ========================================================
# SAVE MODELS
# ========================================================
os.makedirs("/mnt/data/trained_models", exist_ok=True)

for target, model in trained_all.items():
    filepath = f"/mnt/data/trained_models/{target}_catboost_tuned.cbm"
    model.save_model(filepath)
    print(f"💾 Saved tuned CatBoost model for {target} → {filepath}")



# ========================================================
# SUMMARY TABLE
# ========================================================
summary_df = pd.DataFrame([
    {"target": k, "mae": v["mae"], "r2": v["r2"], "params": v["best_params"]}
    for k, v in results_all.items()
])

display(summary_df)
print("\n✅ All CatBoost models tuned, trained, evaluated, and saved.")



 TRAINING WITH HYPERTUNING → demand
🔍 Hyper-tuning CatBoost for demand ...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
➡ Best Params: {'learning_rate': 0.03, 'l2_leaf_reg': 1, 'iterations': 800, 'depth': 4, 'bagging_temperature': 0.1}

📌 Performance for demand:
MAE: 3.7451029690735793
R2: 0.3074294268659905

 TRAINING WITH HYPERTUNING → available_cabs
🔍 Hyper-tuning CatBoost for available_cabs ...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
➡ Best Params: {'learning_rate': 0.03, 'l2_leaf_reg': 3, 'iterations': 300, 'depth': 4, 'bagging_temperature': 1}

📌 Performance for available_cabs:
MAE: 4.300157949485737
R2: 0.1217132493517844
💾 Saved tuned CatBoost model for demand → /mnt/data/trained_models/demand_catboost_tuned.cbm
💾 Saved tuned CatBoost model for available_cabs → /mnt/data/trained_models/available_cabs_catboost_tuned.cbm


,target,mae,r2,params
0,demand,3.745103,0.307429,"{'learning_rate': 0.03, 'l2_leaf_reg': 1, 'ite..."
1,available_cabs,4.300158,0.121713,"{'learning_rate': 0.03, 'l2_leaf_reg': 3, 'ite..."



✅ All CatBoost models tuned, trained, evaluated, and saved.


In [24]:
# ========================================================
# COMPARISON TABLE + BEST MODEL SELECTION
# ========================================================

comparison_df = pd.DataFrame([
    {
        "target": target,
        "MAE": metrics["mae"],
        "R2": metrics["r2"],
        "Best_Params": metrics["best_params"]
    }
    for target, metrics in results_all.items()
])

print("\n📌 Final Comparison Table:")
display(comparison_df)

# --------------------------------------------------------
# SELECT BEST MODEL BASED ON CHOICE
# --------------------------------------------------------

# 1. Based on MAE (lower is better)
best_by_mae = comparison_df.loc[comparison_df['MAE'].idxmin()]

# 2. Based on R2 (higher is better)
best_by_r2 = comparison_df.loc[comparison_df['R2'].idxmax()]

print("\n🏆 Best Model Based on MAE (lower is better):")
display(best_by_mae.to_frame().T)

print("\n🏆 Best Model Based on R2 (higher is better):")
display(best_by_r2.to_frame().T)



📌 Final Comparison Table:


,target,MAE,R2,Best_Params
0,demand,3.745103,0.307429,"{'learning_rate': 0.03, 'l2_leaf_reg': 1, 'ite..."
1,available_cabs,4.300158,0.121713,"{'learning_rate': 0.03, 'l2_leaf_reg': 3, 'ite..."



🏆 Best Model Based on MAE (lower is better):


,target,MAE,R2,Best_Params
0,demand,3.745103,0.307429,"{'learning_rate': 0.03, 'l2_leaf_reg': 1, 'ite..."



🏆 Best Model Based on R2 (higher is better):


,target,MAE,R2,Best_Params
0,demand,3.745103,0.307429,"{'learning_rate': 0.03, 'l2_leaf_reg': 1, 'ite..."


In [25]:
from catboost import CatBoostRegressor

model_demand = CatBoostRegressor()
model_demand.load_model('/mnt/data/trained_models/demand_catboost_tuned.cbm')

model_cabs = CatBoostRegressor()
model_cabs.load_model('/mnt/data/trained_models/available_cabs_catboost_tuned.cbm')
